# Stage 2: RoBERTa Stance Model Training

**執行前請先確認：Runtime → Change runtime type → T4 GPU**

Checkpoint 會自動存到 Google Drive，斷線重連後可以接著跑。

In [ ]:
# 掛載 Google Drive（checkpoint 存這裡，斷線不會消失）
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = '/content/drive/MyDrive/stance_checkpoints'
import os
os.makedirs(CKPT_DIR, exist_ok=True)
print(f'Checkpoint dir: {CKPT_DIR}')

In [ ]:
!pip install -q transformers datasets scikit-learn

In [ ]:
import math, json
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from transformers import RobertaTokenizer, RobertaModel
from datasets import load_dataset
from sklearn.model_selection import GroupKFold

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory/1024**3:.1f} GB')

In [ ]:
# ── 參數 ──
ROBERTA_NAME = 'roberta-base'
EPOCHS       = 3
LR           = 2e-5
BATCH_SIZE   = 32
MAX_LENGTH   = 256
N_FOLDS      = 4
MIN_CONF     = 0.8

# 設為 True 可跳過 CV，直接訓練最終模型（省時間，適合 Colab 免費版）
SKIP_CV      = False

In [ ]:
# ── 載入資料集 ──
print('Loading dataset...')
raw = load_dataset('ibm-research/argument_quality_ranking_30k', 'argument_quality_ranking')

all_rows = []
for split in raw.values():
    for row in split:
        if float(row['stance_WA_conf']) < MIN_CONF:
            continue
        all_rows.append({
            'topic':    row['topic'],
            'argument': row['argument'],
            'stance':   float(row['stance_WA']),
        })

n_topics = len(set(r['topic'] for r in all_rows))
print(f'Loaded {len(all_rows):,} examples, {n_topics} topics')

In [ ]:
# ── Dataset & Model ──
class StanceDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=256):
        self.samples   = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        s   = self.samples[idx]
        enc = self.tokenizer(
            s['topic'], s['argument'],
            max_length=self.max_length,
            truncation=True, padding='max_length',
            return_tensors='pt',
        )
        return {
            'input_ids':      enc['input_ids'].squeeze(0),
            'attention_mask': enc['attention_mask'].squeeze(0),
            'stance':         torch.tensor(s['stance'], dtype=torch.float32),
        }


class StanceModel(nn.Module):
    def __init__(self, name='roberta-base'):
        super().__init__()
        self.roberta   = RobertaModel.from_pretrained(name)
        self.regressor = nn.Linear(768, 1)

    def forward(self, input_ids, attention_mask):
        cls = self.roberta(input_ids=input_ids,
                           attention_mask=attention_mask).last_hidden_state[:, 0, :]
        return torch.tanh(self.regressor(cls)).squeeze(-1)


def train_epoch(model, loader, optimizer, scheduler, criterion):
    model.train()
    total = 0.0
    for batch in loader:
        ids  = batch['input_ids'].to(device)
        mask = batch['attention_mask'].to(device)
        y    = batch['stance'].to(device)
        optimizer.zero_grad()
        loss = criterion(model(ids, mask), y)
        loss.backward()
        nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        scheduler.step()
        total += loss.item()
    return total / len(loader)


def evaluate(model, loader):
    model.eval()
    preds, labels = [], []
    with torch.no_grad():
        for batch in loader:
            p = model(batch['input_ids'].to(device),
                      batch['attention_mask'].to(device)).cpu().numpy()
            preds.extend(p.tolist())
            labels.extend(batch['stance'].numpy().tolist())
    mse = float(np.mean((np.array(preds) - np.array(labels)) ** 2))
    mae = float(np.mean(np.abs(np.array(preds) - np.array(labels))))
    return mse, mae

In [ ]:
tokenizer     = RobertaTokenizer.from_pretrained(ROBERTA_NAME)
topics        = [r['topic'] for r in all_rows]
unique_topics = sorted(set(topics))
topic_to_idx  = {t: i for i, t in enumerate(unique_topics)}
groups        = np.array([topic_to_idx[t] for t in topics])
criterion     = nn.MSELoss()
fold_results  = [None] * N_FOLDS  # 固定長度，支援 index 賦值

if not SKIP_CV:
    # ── 檢查已完成的 fold（斷線重連後自動跳過）──
    completed_folds = set()
    for fname in os.listdir(CKPT_DIR):
        if fname.startswith('fold') and fname.endswith('.pt'):
            fold_num = int(fname[4])
            ckpt = torch.load(os.path.join(CKPT_DIR, fname), map_location='cpu')
            fold_results[fold_num - 1] = ckpt['val_mse']
            completed_folds.add(fold_num)
            print(f'  Found existing Fold {fold_num} checkpoint (MSE={ckpt["val_mse"]:.4f})')

    gkf = GroupKFold(n_splits=N_FOLDS)
    for fold, (train_idx, val_idx) in enumerate(
        gkf.split(all_rows, groups=groups), 1
    ):
        if fold in completed_folds:
            print(f'Fold {fold} already done, skipping.')
            continue

        print(f'\n--- Fold {fold}/{N_FOLDS}  ({len(val_idx)} val examples) ---')
        train_loader = DataLoader(
            StanceDataset([all_rows[i] for i in train_idx], tokenizer, MAX_LENGTH),
            batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
        )
        val_loader = DataLoader(
            StanceDataset([all_rows[i] for i in val_idx], tokenizer, MAX_LENGTH),
            batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
        )

        model     = StanceModel(ROBERTA_NAME).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=0.01)
        scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer, T_max=EPOCHS * len(train_loader)
        )
        best_mse, best_state = float('inf'), None

        for epoch in range(1, EPOCHS + 1):
            train_loss       = train_epoch(model, train_loader, optimizer, scheduler, criterion)
            val_mse, val_mae = evaluate(model, val_loader)
            print(f'  Epoch {epoch}/{EPOCHS}  train={train_loss:.4f}  '
                  f'val_mse={val_mse:.4f}  val_mae={val_mae:.4f}')
            if val_mse < best_mse:
                best_mse   = val_mse
                best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

        fold_results[fold - 1] = best_mse
        ckpt_path = os.path.join(CKPT_DIR, f'fold{fold}_mse{best_mse:.4f}.pt')
        torch.save({'model_state': best_state, 'roberta_name': ROBERTA_NAME,
                    'val_mse': best_mse, 'fold': fold}, ckpt_path)
        print(f'  Saved → {ckpt_path}')

    fold_results = [x for x in fold_results if x is not None]
    mean_mse = float(np.mean(fold_results))
    print(f'\n=== {N_FOLDS}-fold CV ===')
    for i, mse in enumerate(fold_results, 1):
        print(f'  Fold {i}: MSE={mse:.4f}  RMSE={math.sqrt(mse):.4f}')
    print(f'  Mean MSE={mean_mse:.4f}  RMSE={math.sqrt(mean_mse):.4f}')
else:
    mean_mse = None
    print('SKIP_CV=True，跳過交叉驗證，直接訓練最終模型')

In [ ]:
# ── 最終模型（全部資料）──
print('\nTraining final model on all data...')
full_loader = DataLoader(
    StanceDataset(all_rows, tokenizer, MAX_LENGTH),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
)
final_model = StanceModel(ROBERTA_NAME).to(device)
optimizer   = torch.optim.AdamW(final_model.parameters(), lr=LR, weight_decay=0.01)
scheduler   = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS * len(full_loader)
)

for epoch in range(1, EPOCHS + 1):
    loss = train_epoch(final_model, full_loader, optimizer, scheduler, criterion)
    print(f'  Epoch {epoch}/{EPOCHS}  loss={loss:.4f}')

final_path = os.path.join(CKPT_DIR, 'final_model.pt')
torch.save({
    'model_state':  {k: v.cpu() for k, v in final_model.state_dict().items()},
    'roberta_name': ROBERTA_NAME,
    'cv_mean_mse':  mean_mse,
    'cv_fold_mses': fold_results,
}, final_path)

cv_path = os.path.join(CKPT_DIR, 'cv_results.json')
with open(cv_path, 'w') as f:
    json.dump({'fold_mses': fold_results, 'mean_mse': mean_mse,
               'mean_rmse': math.sqrt(mean_mse) if mean_mse else None}, f, indent=2)

print(f'\nfinal_model.pt saved → {final_path}')

In [ ]:
# ── 下載到本機 ──
from google.colab import files
files.download(final_path)
files.download(cv_path)